# BART Ridership Data Audit

This notebook audits the processed ridership pipeline used by the Dash app. The key data decision is:

```text
Hourly OD is the canonical ridership source where available.
Workbook Total Trips / Total Trips OD files are validation benchmarks and fallback sources.
```

The notebook focuses on station-code normalization, source coverage, and differences between hourly OD and workbook totals.


In [37]:
from pathlib import Path
import sys

import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)


def find_project_root(start=None):
    """Find the repo root from either notebooks/ or the repo root itself."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing src/ and data/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED = PROJECT_ROOT / "data" / "processed"
RAW = PROJECT_ROOT / "data" / "raw"
HOURLY_OD_DIR = RAW / "Hourly_OD"

PROJECT_ROOT


WindowsPath('c:/Users/Carl Wheezer/Documents/git-test/bart-ridership-dash')

## Processed Data Inputs

The app should read processed files only. Raw hourly OD CSVs and workbooks are used by `scripts/prepare_data.py` to regenerate these artifacts.


In [38]:
app_summary_path = PROCESSED / "station_ridership_monthly_summary.csv"
hourly_summary_path = PROCESSED / "hourly_od_station_monthly_summary.csv"
validation_path = PROCESSED / "hourly_od_total_trips_validation.csv"

processed_files_df = pd.DataFrame(
    [
        {
            "Artifact": path.name,
            "Path": path.relative_to(PROJECT_ROOT).as_posix(),
            "Exists": path.exists(),
            "Size MB": path.stat().st_size / 1_000_000 if path.exists() else None,
        }
        for path in [app_summary_path, hourly_summary_path, validation_path]
    ]
)
processed_files_df


,Artifact,Path,Exists,Size MB
0,station_ridership_monthly_summary.csv,data/processed/station_ridership_monthly_summa...,True,0.75
1,hourly_od_station_monthly_summary.csv,data/processed/hourly_od_station_monthly_summa...,True,0.75
2,hourly_od_total_trips_validation.csv,data/processed/hourly_od_total_trips_validatio...,True,1.35


In [39]:
app_summary_df = pd.read_csv(app_summary_path)
hourly_summary_df = pd.read_csv(hourly_summary_path)
valid_ridership_df = pd.read_csv(validation_path)

summary_shapes_df = pd.DataFrame(
    [
        {"Table": "App monthly station summary", "Rows": len(app_summary_df), "Columns": len(app_summary_df.columns)},
        {"Table": "Hourly OD monthly station summary", "Rows": len(hourly_summary_df), "Columns": len(hourly_summary_df.columns)},
        {"Table": "Hourly OD vs workbook validation", "Rows": len(valid_ridership_df), "Columns": len(valid_ridership_df.columns)},
    ]
)
summary_shapes_df


,Table,Rows,Columns
0,App monthly station summary,4755,8
1,Hourly OD monthly station summary,4755,8
2,Hourly OD vs workbook validation,4741,12


In [40]:
period_coverage_df = (
    app_summary_df
    .groupby(["Year", "Month", "Period", "Source Type"], as_index=False)
    .agg(
        Stations=("Entry Station", "nunique"),
        Total_Ridership=("Ridership", "sum"),
    )
    .sort_values(["Year", "Month"])
)

period_coverage_df.head(), period_coverage_df.tail()


(   Year  Month   Period Source Type  Stations  Total_Ridership
 0  2018      1  2018-01   hourly_od        46          9785965
 1  2018      2  2018-02   hourly_od        46          9262187
 2  2018      3  2018-03   hourly_od        47         10364615
 3  2018      4  2018-04   hourly_od        48         10049472
 4  2018      5  2018-05   hourly_od        48         10595609,
     Year  Month   Period Source Type  Stations  Total_Ridership
 91  2025      8  2025-08   hourly_od        50          4908797
 92  2025      9  2025-09   hourly_od        50          4724313
 93  2025     10  2025-10   hourly_od        50          4930734
 94  2025     11  2025-11   hourly_od        50          4031002
 95  2025     12  2025-12   hourly_od        50          3768928)

In [41]:
source_type_summary_df = (
    app_summary_df
    .groupby("Source Type", as_index=False)
    .agg(
        Rows=("Entry Station", "count"),
        Periods=("Period", "nunique"),
        Total_Ridership=("Ridership", "sum"),
    )
)
source_type_summary_df


,Source Type,Rows,Periods,Total_Ridership
0,hourly_od,4755,96,492978618


## January 2018 Workbook Edge Case

`Ridership_201801.xlsx` does not contain a comparable monthly Total Trips sheet. It contains average weekday/Saturday/Sunday OD sheets, so January 2018 is taken from hourly OD instead.


In [42]:
jan_2018_workbook = RAW / "ridership_2018" / "Ridership_201801.xlsx"
jan_2018_hourly = HOURLY_OD_DIR / "date-hour-soo-dest-2018.csv"

jan_2018_sheets = pd.ExcelFile(jan_2018_workbook).sheet_names
jan_2018_sheets


['Weekday OD', 'Saturday OD', 'Sunday OD']

In [43]:
def hourly_month_total(csv_path, year, month, chunksize=500_000):
    """Compute one monthly total without loading the full hourly CSV into memory."""
    period_prefix = f"{year}-{month:02d}"
    total = 0
    for chunk in pd.read_csv(
        csv_path,
        header=None,
        names=["Date", "Hour", "Origin", "Destination", "Exits"],
        usecols=["Date", "Exits"],
        chunksize=chunksize,
    ):
        selected = chunk[chunk["Date"].astype(str).str.startswith(period_prefix)]
        total += pd.to_numeric(selected["Exits"], errors="coerce").fillna(0).sum()
    return int(total)

jan_2018_hourly_total = hourly_month_total(jan_2018_hourly, 2018, 1)
jan_2018_app_total = int(
    app_summary_df.loc[(app_summary_df["Year"] == 2018) & (app_summary_df["Month"] == 1), "Ridership"].sum()
)

pd.DataFrame(
    [
        {"Metric": "January 2018 hourly OD total", "Ridership": jan_2018_hourly_total},
        {"Metric": "January 2018 app summary total", "Ridership": jan_2018_app_total},
    ]
)


,Metric,Ridership
0,January 2018 hourly OD total,9785965
1,January 2018 app summary total,9785965


## Station Code Alias Audit

This table documents every explicit station-code alias used by the pipeline. The goal is to make source-specific idiosyncrasies visible instead of hiding them inside parser logic.


In [55]:
from src.station_mapping import (
    CANONICAL_CODE_ALIASES,
    HOURLY_OD_STATION_ALIASES,
    IGNORED_STATION_NAMES,
    STATION_MAPPING,
    WORKBOOK_STATION_ALIASES,
)


def alias_rows(alias_map, source_system, note):
    rows = []
    for raw_code, canonical_code in sorted(alias_map.items()):
        rows.append(
            {
                "Source System": source_system,
                "Raw Label / Code": raw_code,
                "Canonical Code": canonical_code,
                "Canonical Station Name": STATION_MAPPING.get(canonical_code),
                "Changes Code?": raw_code != canonical_code,
                "Audit Note": note,
            }
        )
    return rows


alias_audit_df = pd.DataFrame(
    alias_rows(CANONICAL_CODE_ALIASES, "General canonical cleanup", "Normalizes numeric station labels from workbook headers.")
    + alias_rows(HOURLY_OD_STATION_ALIASES, "Hourly OD CSV", "Normalizes four-character hourly OD station codes into canonical app codes.")
    + alias_rows(WORKBOOK_STATION_ALIASES, "Workbook Total Trips", "Normalizes legacy workbook labels that conflict with canonical app codes.")
    + [
        {
            "Source System": "Station geometry KML",
            "Raw Label / Code": name,
            "Canonical Code": None,
            "Canonical Station Name": None,
            "Changes Code?": True,
            "Audit Note": "Ignored for ridership station display; geometry/infrastructure marker, not a station ridership code.",
        }
        for name in sorted(IGNORED_STATION_NAMES)
    ]
)

alias_audit_df


,Source System,Raw Label / Code,Canonical Code,Canonical Station Name,Changes Code?,Audit Note
0,General canonical cleanup,12,12th,12th St/Oakland City Center,True,Normalizes numeric station labels from workboo...
1,General canonical cleanup,12.0,12th,12th St/Oakland City Center,True,Normalizes numeric station labels from workboo...
2,General canonical cleanup,16,16th,16th St/Mission,True,Normalizes numeric station labels from workboo...
3,General canonical cleanup,16.0,16th,16th St/Mission,True,Normalizes numeric station labels from workboo...
4,General canonical cleanup,19,19th,19th St/Oakland,True,Normalizes numeric station labels from workboo...
5,General canonical cleanup,19.0,19th,19th St/Oakland,True,Normalizes numeric station labels from workboo...
6,General canonical cleanup,24,24th,24th St/Mission,True,Normalizes numeric station labels from workboo...
7,General canonical cleanup,24.0,24th,24th St/Mission,True,Normalizes numeric station labels from workboo...
8,Hourly OD CSV,12TH,12th,12th St/Oakland City Center,True,Normalizes four-character hourly OD station co...
9,Hourly OD CSV,16TH,16th,16th St/Mission,True,Normalizes four-character hourly OD station co...


In [45]:
# If this table is non-empty, an alias points to a station code missing from STATION_MAPPING.
alias_target_holes_df = alias_audit_df[
    alias_audit_df["Canonical Code"].notna()
    & alias_audit_df["Canonical Station Name"].isna()
].copy()

alias_target_holes_df


,Source System,Raw Label / Code,Canonical Code,Canonical Station Name,Changes Code?,Audit Note


In [46]:
alias_changes_only_df = alias_audit_df[alias_audit_df["Changes Code?"]].copy()
alias_changes_only_df


,Source System,Raw Label / Code,Canonical Code,Canonical Station Name,Changes Code?,Audit Note
0,General canonical cleanup,12,12th,12th St/Oakland City Center,True,Normalizes numeric station labels from workboo...
1,General canonical cleanup,12.0,12th,12th St/Oakland City Center,True,Normalizes numeric station labels from workboo...
2,General canonical cleanup,16,16th,16th St/Mission,True,Normalizes numeric station labels from workboo...
3,General canonical cleanup,16.0,16th,16th St/Mission,True,Normalizes numeric station labels from workboo...
4,General canonical cleanup,19,19th,19th St/Oakland,True,Normalizes numeric station labels from workboo...
5,General canonical cleanup,19.0,19th,19th St/Oakland,True,Normalizes numeric station labels from workboo...
6,General canonical cleanup,24,24th,24th St/Mission,True,Normalizes numeric station labels from workboo...
7,General canonical cleanup,24.0,24th,24th St/Mission,True,Normalizes numeric station labels from workboo...
8,Hourly OD CSV,12TH,12th,12th St/Oakland City Center,True,Normalizes four-character hourly OD station co...
9,Hourly OD CSV,16TH,16th,16th St/Mission,True,Normalizes four-character hourly OD station co...


## Raw Hourly OD Code Coverage

This checks the raw hourly OD station codes found in each yearly CSV against the known alias table. Any `Unmapped Raw Codes` value should be investigated before treating hourly OD as canonical for that year.


In [47]:
def raw_hourly_codes_for_file(csv_path, chunksize=500_000):
    codes = set()
    for chunk in pd.read_csv(
        csv_path,
        header=None,
        names=["Date", "Hour", "Origin", "Destination", "Exits"],
        usecols=["Origin", "Destination"],
        chunksize=chunksize,
    ):
        codes.update(chunk["Origin"].dropna().astype(str).unique())
        codes.update(chunk["Destination"].dropna().astype(str).unique())
    return codes


known_hourly_codes = set(HOURLY_OD_STATION_ALIASES)
hourly_code_coverage_rows = []
for csv_path in sorted(HOURLY_OD_DIR.glob("date-hour-soo-dest-*.csv")):
    year = int(csv_path.stem.split("-")[-1])
    raw_codes = raw_hourly_codes_for_file(csv_path)
    hourly_code_coverage_rows.append(
        {
            "Year": year,
            "Raw Code Count": len(raw_codes),
            "Unmapped Raw Codes": sorted(raw_codes - known_hourly_codes),
            "Mapped Raw Codes": sorted(raw_codes & known_hourly_codes),
        }
    )

hourly_code_coverage_df = pd.DataFrame(hourly_code_coverage_rows)
hourly_code_coverage_df[["Year", "Raw Code Count", "Unmapped Raw Codes"]]


,Year,Raw Code Count,Unmapped Raw Codes
0,2018,48,[]
1,2019,50,[]
2,2020,50,[]
3,2021,50,[]
4,2022,50,[]
5,2023,50,[]
6,2024,50,[]
7,2025,50,[]


## Hourly OD vs Workbook Validation

The validation table compares station-level monthly totals from hourly OD against workbook Total Trips / Total Trips OD. Since hourly OD is canonical where available, workbook values are treated as a benchmark rather than the runtime source of truth.


In [48]:
validation_audit_df = valid_ridership_df.copy()
validation_audit_df["Workbook Ridership"] = pd.to_numeric(validation_audit_df["Workbook Ridership"], errors="coerce").fillna(0)
validation_audit_df["Hourly OD Ridership"] = pd.to_numeric(validation_audit_df["Hourly OD Ridership"], errors="coerce").fillna(0)
validation_audit_df["Difference"] = validation_audit_df["Hourly OD Ridership"] - validation_audit_df["Workbook Ridership"]
validation_audit_df["Absolute Difference"] = validation_audit_df["Difference"].abs()
validation_audit_df["Percent Difference vs Workbook"] = (
    validation_audit_df["Difference"]
    / validation_audit_df["Workbook Ridership"].replace(0, pd.NA)
    * 100
)
validation_audit_df["Absolute Percent Difference vs Workbook"] = validation_audit_df["Percent Difference vs Workbook"].abs()
validation_audit_df["Difference Direction"] = validation_audit_df["Difference"].map(
    lambda value: "Hourly higher" if value > 0 else ("Workbook higher" if value < 0 else "Match")
)


def audit_flag(row):
    if row["Workbook Ridership"] == 0 and row["Hourly OD Ridership"] > 0:
        return "Missing workbook benchmark / workbook station code issue"
    if row["Hourly OD Ridership"] == 0 and row["Workbook Ridership"] > 0:
        return "Missing hourly OD benchmark / hourly station alias issue"
    if row["Absolute Difference"] >= 10_000:
        return "Large absolute difference"
    if pd.notna(row["Absolute Percent Difference vs Workbook"]) and row["Absolute Percent Difference vs Workbook"] >= 5:
        return "Large percentage difference"
    if row["Absolute Difference"] > 0:
        return "Small source-definition difference"
    return "Exact match"


validation_audit_df["Audit Flag"] = validation_audit_df.apply(audit_flag, axis=1)
validation_audit_df = validation_audit_df.sort_values(
    ["Audit Flag", "Absolute Difference"],
    ascending=[True, False],
).reset_index(drop=True)

validation_audit_df


,Year,Month,Period,Entry Station,Full Station Name,Workbook Ridership,Hourly OD Ridership,Difference,Absolute Difference,Has Difference,Workbook Source File,Hourly Source File,Percent Difference vs Workbook,Absolute Percent Difference vs Workbook,Difference Direction,Audit Flag
0,2018,5,2018-05,BE,Berryessa/North San José,0.00,0.00,0.00,0.00,False,C:\Users\Carl Wheezer\Documents\git-test\bart-...,NaN,<NA>,<NA>,Match,Exact match
1,2018,5,2018-05,ML,Milpitas,0.00,0.00,0.00,0.00,False,C:\Users\Carl Wheezer\Documents\git-test\bart-...,NaN,<NA>,<NA>,Match,Exact match
2,2018,6,2018-06,BE,Berryessa/North San José,0.00,0.00,0.00,0.00,False,C:\Users\Carl Wheezer\Documents\git-test\bart-...,NaN,<NA>,<NA>,Match,Exact match
3,2018,6,2018-06,ML,Milpitas,0.00,0.00,0.00,0.00,False,C:\Users\Carl Wheezer\Documents\git-test\bart-...,NaN,<NA>,<NA>,Match,Exact match
4,2018,7,2018-07,BE,Berryessa/North San José,0.00,0.00,0.00,0.00,False,C:\Users\Carl Wheezer\Documents\git-test\bart-...,NaN,<NA>,<NA>,Match,Exact match
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4736,2025,3,2025-03,OA,Oakland International Airport,"17,448.00","17,436.00",-12.00,12.00,True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,-0.07,0.07,Workbook higher,Small source-definition difference
4737,2020,5,2020-05,OA,Oakland International Airport,"1,626.00","1,635.00",9.00,9.00,True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,0.55,0.55,Hourly higher,Small source-definition difference
4738,2020,4,2020-04,LF,Lafayette,"2,298.00","2,304.00",6.00,6.00,True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,0.26,0.26,Hourly higher,Small source-definition difference
4739,2020,4,2020-04,WD,Warm Springs/South Fremont,"4,332.00","4,337.00",5.00,5.00,True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,0.12,0.12,Hourly higher,Small source-definition difference


In [49]:
audit_flag_summary_df = (
    validation_audit_df
    .groupby("Audit Flag", as_index=False)
    .agg(
        Rows=("Period", "count"),
        Periods=("Period", "nunique"),
        Stations=("Entry Station", "nunique"),
        Max_Absolute_Difference=("Absolute Difference", "max"),
        Mean_Absolute_Difference=("Absolute Difference", "mean"),
    )
    .sort_values("Rows", ascending=False)
)
audit_flag_summary_df


,Audit Flag,Rows,Periods,Stations,Max_Absolute_Difference,Mean_Absolute_Difference
4,Small source-definition difference,4155,93,50,"9,949.00","1,359.98"
2,Large percentage difference,393,72,44,"9,951.00","4,242.72"
1,Large absolute difference,140,60,24,"83,755.00","19,076.64"
0,Exact match,32,17,2,0.00,0.00
3,Missing workbook benchmark / workbook station ...,21,12,4,138.00,39.24


In [50]:
priority_validation_rows_df = validation_audit_df[
    validation_audit_df["Audit Flag"].isin(
        [
            "Missing workbook benchmark / workbook station code issue",
            "Missing hourly OD benchmark / hourly station alias issue",
            "Large absolute difference",
            "Large percentage difference",
        ]
    )
].sort_values("Absolute Difference", ascending=False)

priority_validation_rows_df


,Year,Month,Period,Entry Station,Full Station Name,Workbook Ridership,Hourly OD Ridership,Difference,Absolute Difference,Has Difference,Workbook Source File,Hourly Source File,Percent Difference vs Workbook,Absolute Percent Difference vs Workbook,Difference Direction,Audit Flag
32,2020,2,2020-02,MT,Montgomery St,"876,698.00","792,943.00","-83,755.00","83,755.00",True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,-9.55,9.55,Workbook higher,Large absolute difference
33,2020,2,2020-02,EM,Embarcadero,"848,546.00","769,637.00","-78,909.00","78,909.00",True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,-9.30,9.30,Workbook higher,Large absolute difference
34,2025,12,2025-12,EM,Embarcadero,"373,590.00","308,762.00","-64,828.00","64,828.00",True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,-17.35,17.35,Workbook higher,Large absolute difference
35,2025,12,2025-12,PL,Powell St,"352,353.00","291,256.00","-61,097.00","61,097.00",True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,-17.34,17.34,Workbook higher,Large absolute difference
36,2025,12,2025-12,MT,Montgomery St,"303,035.00","254,156.00","-48,879.00","48,879.00",True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,-16.13,16.13,Workbook higher,Large absolute difference
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581,2018,3,2018-03,AN,Antioch,0.00,4.00,4.00,4.00,True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,<NA>,<NA>,Hourly higher,Missing workbook benchmark / workbook station ...
582,2019,12,2019-12,BE,Berryessa/North San José,0.00,2.00,2.00,2.00,True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,<NA>,<NA>,Hourly higher,Missing workbook benchmark / workbook station ...
583,2020,4,2020-04,ML,Milpitas,0.00,2.00,2.00,2.00,True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,<NA>,<NA>,Hourly higher,Missing workbook benchmark / workbook station ...
584,2020,2,2020-02,BE,Berryessa/North San José,0.00,1.00,1.00,1.00,True,C:\Users\Carl Wheezer\Documents\git-test\bart-...,C:\Users\Carl Wheezer\Documents\git-test\bart-...,<NA>,<NA>,Hourly higher,Missing workbook benchmark / workbook station ...


In [51]:
station_validation_summary_df = (
    validation_audit_df
    .groupby(["Entry Station", "Full Station Name"], dropna=False)
    .agg(
        Periods_Compared=("Period", "nunique"),
        Rows_With_Difference=("Has Difference", "sum"),
        Max_Absolute_Difference=("Absolute Difference", "max"),
        Mean_Absolute_Difference=("Absolute Difference", "mean"),
        Total_Absolute_Difference=("Absolute Difference", "sum"),
        Max_Absolute_Percent_Difference=("Absolute Percent Difference vs Workbook", "max"),
        Missing_Hourly_Rows=("Audit Flag", lambda values: (values == "Missing hourly OD benchmark / hourly station alias issue").sum()),
        Missing_Workbook_Rows=("Audit Flag", lambda values: (values == "Missing workbook benchmark / workbook station code issue").sum()),
        Large_Absolute_Difference_Rows=("Audit Flag", lambda values: (values == "Large absolute difference").sum()),
        Large_Percentage_Difference_Rows=("Audit Flag", lambda values: (values == "Large percentage difference").sum()),
    )
    .reset_index()
    .sort_values(["Missing_Hourly_Rows", "Missing_Workbook_Rows", "Max_Absolute_Difference"], ascending=False)
)

station_validation_summary_df


,Entry Station,Full Station Name,Periods_Compared,Rows_With_Difference,Max_Absolute_Difference,Mean_Absolute_Difference,Total_Absolute_Difference,Max_Absolute_Percent_Difference,Missing_Hourly_Rows,Missing_Workbook_Rows,Large_Absolute_Difference_Rows,Large_Percentage_Difference_Rows
6,BE,Berryessa/North San José,92,77,"7,114.00","1,049.74","96,576.00",27.96,0,10,0,32
28,ML,Milpitas,92,75,"7,142.00",691.51,"63,619.00",53.53,0,8,0,15
4,AN,Antioch,94,94,"5,251.00","1,718.27","161,517.00",11.91,0,2,0,23
35,PC,Pittsburg Center,93,93,"2,094.00",424.60,"39,488.00",9.70,0,1,0,4
29,MT,Montgomery St,95,95,"83,755.00","5,444.57","517,234.00",16.13,0,0,6,0
17,EM,Embarcadero,95,95,"78,909.00","6,584.82","625,558.00",17.35,0,0,7,0
38,PL,Powell St,95,95,"61,097.00","6,256.27","594,346.00",17.34,0,0,6,0
10,CC,Civic Center/UN Plaza,95,95,"44,754.00","5,757.72","546,983.00",13.99,0,0,6,0
44,SO,San Francisco International Airport,95,95,"26,789.00","7,648.79","726,635.00",20.39,0,0,34,13
0,12th,12th St/Oakland City Center,95,95,"25,639.00","2,768.67","263,024.00",12.74,0,0,4,2


## Period-Level Source Differences

Period-level differences reveal months where the two sources diverge broadly, which is different from a single station-code mapping issue.


In [52]:
period_validation_summary_df = (
    validation_audit_df
    .groupby(["Year", "Month", "Period"], as_index=False)
    .agg(
        Workbook_Total=("Workbook Ridership", "sum"),
        Hourly_OD_Total=("Hourly OD Ridership", "sum"),
        Total_Absolute_Station_Difference=("Absolute Difference", "sum"),
        Max_Station_Difference=("Absolute Difference", "max"),
        Stations_Compared=("Entry Station", "nunique"),
        Priority_Row_Count=(
            "Audit Flag",
            lambda values: values.isin(
                [
                    "Missing workbook benchmark / workbook station code issue",
                    "Missing hourly OD benchmark / hourly station alias issue",
                    "Large absolute difference",
                    "Large percentage difference",
                ]
            ).sum(),
        ),
    )
)
period_validation_summary_df["Total Difference"] = period_validation_summary_df["Hourly_OD_Total"] - period_validation_summary_df["Workbook_Total"]
period_validation_summary_df["Total Percent Difference vs Workbook"] = (
    period_validation_summary_df["Total Difference"]
    / period_validation_summary_df["Workbook_Total"].replace(0, pd.NA)
    * 100
)
period_validation_summary_df = period_validation_summary_df.sort_values("Max_Station_Difference", ascending=False)
period_validation_summary_df


,Year,Month,Period,Workbook_Total,Hourly_OD_Total,Total_Absolute_Station_Difference,Max_Station_Difference,Stations_Compared,Priority_Row_Count,Total Difference,Total Percent Difference vs Workbook
24,2020,2,2020-02,"8,976,810.00","8,244,817.00","731,995.00","83,755.00",50,49,"-731,993.00",-8.15
94,2025,12,2025-12,"4,395,799.00","3,768,928.00","626,871.00","64,828.00",50,50,"-626,871.00",-14.26
92,2025,10,2025-10,"5,346,891.00","4,930,734.00","416,207.00","44,749.00",50,39,"-416,157.00",-7.78
93,2025,11,2025-11,"4,409,065.00","4,031,002.00","378,063.00","39,498.00",50,41,"-378,063.00",-8.57
23,2020,1,2020-01,"9,358,806.00","9,047,397.00","311,409.00","37,713.00",50,8,"-311,409.00",-3.33
91,2025,9,2025-09,"5,047,121.00","4,724,313.00","323,310.00","36,370.00",50,28,"-322,808.00",-6.40
80,2024,10,2024-10,"4,831,431.00","4,941,039.00","109,608.00","20,724.00",50,4,"109,608.00",2.27
83,2025,1,2025-01,"4,151,331.00","4,261,168.00","109,837.00","20,052.00",50,4,"109,837.00",2.65
89,2025,7,2025-07,"4,713,700.00","4,823,002.00","109,302.00","19,901.00",50,5,"109,302.00",2.32
82,2024,12,2024-12,"3,874,228.00","3,964,739.00","92,769.00","19,813.00",50,4,"90,511.00",2.34


## Processed Source Coverage Audit

The app summary should be hourly-derived for every available hourly OD period. This table checks station-period coverage across the app summary, hourly summary, and workbook benchmark.


In [53]:
app_keys = app_summary_df[["Year", "Month", "Period", "Entry Station", "Full Station Name", "Source Type"]].copy()
app_keys["In App Summary"] = True

hourly_keys = hourly_summary_df[["Year", "Month", "Period", "Entry Station", "Full Station Name"]].copy()
hourly_keys["In Hourly Summary"] = True

# Only nonzero benchmark rows are used for coverage-hole detection.
# Some workbooks include unopened future stations as explicit zero rows.
workbook_keys = validation_audit_df[
    (validation_audit_df["Workbook Ridership"] > 0)
    | (validation_audit_df["Hourly OD Ridership"] > 0)
][["Year", "Month", "Period", "Entry Station", "Full Station Name"]].copy()
workbook_keys["In Workbook Benchmark"] = True

coverage_audit_df = (
    app_keys
    .merge(hourly_keys, on=["Year", "Month", "Period", "Entry Station", "Full Station Name"], how="outer")
    .merge(workbook_keys, on=["Year", "Month", "Period", "Entry Station", "Full Station Name"], how="outer")
)
for column in ["In App Summary", "In Hourly Summary", "In Workbook Benchmark"]:
    coverage_audit_df[column] = coverage_audit_df[column].fillna(False)


def coverage_flag(row):
    if not row["In App Summary"]:
        return "Missing from app summary"
    if not row["In Hourly Summary"]:
        return "Missing from hourly summary"
    if not row["In Workbook Benchmark"]:
        return "No workbook benchmark row"
    return "Covered"


coverage_audit_df["Coverage Flag"] = coverage_audit_df.apply(coverage_flag, axis=1)
coverage_audit_df.sort_values(["Coverage Flag", "Year", "Month", "Entry Station"])


,Year,Month,Period,Entry Station,Full Station Name,Source Type,In App Summary,In Hourly Summary,In Workbook Benchmark,Coverage Flag
46,2018,2,2018-02,12th,12th St/Oakland City Center,hourly_od,True,True,True,Covered
47,2018,2,2018-02,16th,16th St/Mission,hourly_od,True,True,True,Covered
48,2018,2,2018-02,19th,19th St/Oakland,hourly_od,True,True,True,Covered
49,2018,2,2018-02,24th,24th St/Mission,hourly_od,True,True,True,Covered
50,2018,2,2018-02,AS,Ashby,hourly_od,True,True,True,Covered
...,...,...,...,...,...,...,...,...,...,...
41,2018,1,2018-01,SS,South San Francisco,hourly_od,True,True,False,No workbook benchmark row
42,2018,1,2018-01,UC,Union City,hourly_od,True,True,False,No workbook benchmark row
43,2018,1,2018-01,WC,Walnut Creek,hourly_od,True,True,False,No workbook benchmark row
44,2018,1,2018-01,WD,Warm Springs/South Fremont,hourly_od,True,True,False,No workbook benchmark row


In [54]:
coverage_holes_df = coverage_audit_df[coverage_audit_df["Coverage Flag"] != "Covered"].copy()
coverage_holes_df.sort_values(["Coverage Flag", "Year", "Month", "Entry Station"])


,Year,Month,Period,Entry Station,Full Station Name,Source Type,In App Summary,In Hourly Summary,In Workbook Benchmark,Coverage Flag
0,2018,1,2018-01,12th,12th St/Oakland City Center,hourly_od,True,True,False,No workbook benchmark row
1,2018,1,2018-01,16th,16th St/Mission,hourly_od,True,True,False,No workbook benchmark row
2,2018,1,2018-01,19th,19th St/Oakland,hourly_od,True,True,False,No workbook benchmark row
3,2018,1,2018-01,24th,24th St/Mission,hourly_od,True,True,False,No workbook benchmark row
4,2018,1,2018-01,AS,Ashby,hourly_od,True,True,False,No workbook benchmark row
5,2018,1,2018-01,BF,Bay Fair,hourly_od,True,True,False,No workbook benchmark row
6,2018,1,2018-01,BK,Downtown Berkeley,hourly_od,True,True,False,No workbook benchmark row
7,2018,1,2018-01,BP,Balboa Park,hourly_od,True,True,False,No workbook benchmark row
8,2018,1,2018-01,CC,Civic Center/UN Plaza,hourly_od,True,True,False,No workbook benchmark row
9,2018,1,2018-01,CL,Coliseum,hourly_od,True,True,False,No workbook benchmark row


## Audit Checklist

Use this checklist before changing the app's source-of-truth rules or adding Parquet/time-lapse features.

- `alias_target_holes_df` should be empty.
- `hourly_code_coverage_df["Unmapped Raw Codes"]` should be empty for every year.
- `coverage_holes_df` should contain only expected benchmark gaps, especially Jan 2018 with no comparable workbook Total Trips sheet.
- `priority_validation_rows_df` should be reviewed and categorized before claiming workbook/hourly agreement.
- Periods with large total percent differences should be investigated separately from station-code mapping issues.
